# 02 — Leak-Free Preprocessing, Zero-Variance Scaling & Sliding Window Generation
**Course:** BCSE209P — Machine Learning Lab  
**Students:** P Naga Venkata Kushal (23BDS0133) & Koteru Tarun (23BDS0103)  
**Faculty Guide:** Dr. Dega Nagaraju  
**Repository:** [Reliability-Aware Anomaly Detection](https://github.com/nvk321/Reliability-Aware-Anomaly-Detection)

---
### Objectives
1. Demonstrate the leak-free data processing pipeline using `src.preprocessing.scaler.RobustStandardScaler`.
2. Address the zero-variance actuator trap with $\epsilon = 10^{-8}$ variance clamping.
3. Demonstrate chronological sliding window generation with `src.preprocessing.windowing.SlidingWindowGenerator`.
4. Verify label aggregation and temporal boundary conditions.


In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.scaler import RobustStandardScaler
from src.preprocessing.windowing import SlidingWindowGenerator
from src.visualization.plots import apply_scientific_style
apply_scientific_style()

print('Preprocessing modules imported successfully!')


### 1. Robust Scaling with Zero-Variance Protection
In industrial CPS datasets, certain backup pumps and valves remain permanently closed or open during training (`var = 0`).
Standard `StandardScaler` from scikit-learn divides by zero, producing `NaN` or `inf` values.
`RobustStandardScaler` implements:
$$\tilde{x}_i(t) = \frac{x_i(t) - \mu_i}{\sigma_i + \epsilon}$$
where $\sigma_i$ is clamped to $1.0$ if $\sigma_i < \epsilon$ ($10^{-8}$).


In [ ]:
# Demonstrate RobustStandardScaler on synthetic test array with constant channel
np.random.seed(42)
synthetic_train = np.random.normal(50.0, 5.0, (1000, 5))
synthetic_train[:, 2] = 0.0  # Channel 2 is permanently off (constant actuator)

scaler = RobustStandardScaler(eps=1e-8)
scaled_train = scaler.fit_transform(synthetic_train)

print('Original Channel Means: ', synthetic_train.mean(axis=0).round(2))
print('Original Channel Stds:  ', synthetic_train.std(axis=0).round(2))
print('Scaled Channel Means:   ', scaled_train.mean(axis=0).round(2))
print('Scaled Channel Stds:    ', scaled_train.std(axis=0).round(2))
print('Has NaNs:               ', np.isnan(scaled_train).any())


### 2. Inspect Fitted Scaler Parameters on SWaT
Let's load the actual saved parameters fitted strictly on the SWaT training split.


In [ ]:
params_path = os.path.join('..', 'data', 'processed', 'SWaT', 'scaler_params.json')
with open(params_path, 'r') as f:
    scaler_params = json.load(f)

print(f"Fitted Sensors: {len(scaler_params['sensor_names'])}")
print(f"Mean of LIT101: {scaler_params['mean'][scaler_params['sensor_names'].index('LIT101')]:.3f}")
print(f"Std of LIT101:  {scaler_params['scale'][scaler_params['sensor_names'].index('LIT101')]:.3f}")
print(f"Scale of P202 (constant actuator): {scaler_params['scale'][scaler_params['sensor_names'].index('P202')]:.3f} (protected from division by zero)")


### 3. Sliding Window Generation
We convert 2D multivariate time-series $(T, M)$ into 3D sequential sliding windows $(N, L, M)$ with window length $L=30$ and stride $S=1$.
Label mode `point_end` assigns the window's label to the ground-truth value at the final time step $t$.


In [ ]:
L = 30
win_gen = SlidingWindowGenerator(window_size=L, stride=1, label_mode='point_end')

# Test on dummy sequence
dummy_x = np.random.randn(100, 5)
dummy_y = np.zeros(100, dtype=int)
dummy_y[50:60] = 1  # 10 anomalous steps

windows, window_labels = win_gen.create_windows(dummy_x, dummy_y)
print(f"Input shape:  {dummy_x.shape}")
print(f"Output shape: {windows.shape} (N, L, M)")
print(f"Labels shape: {window_labels.shape} (Attacks: {window_labels.sum()})")


### Viva Voce Key Takeaways — Section B: Preprocessing
1. **Leak-Free Guarantee:** Normalization statistics $(\mu, \sigma)$ are computed **strictly on the normal training split**. Validation and test sets are transformed using frozen train statistics.
2. **Zero-Variance Resilience:** Any sensor or actuator with zero natural variance during training has its scale clamped to $1.0$, preventing runtime floating-point errors.
3. **Sliding Window Alignment:** Windows of size $L=30$ ($300\text{ s} = 5\text{ min}$ of historical context) provide sufficient temporal trajectory to forecast the next observation.
